In [1]:
import sys
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import struct


import funciones_aux as fau

import funciones_dsa as fun_dsa
import funciones_dsa_unilateral as fun_dsa_u
import funciones_dsa_bilateral as fun_dsa_b

import funciones_plot_dsa as fun_plot

from scipy.signal import welch
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy.stats import pearsonr, spearmanr


In [2]:
from scipy.ndimage import gaussian_filter1d
import numpy as np
import pandas as pd


def aplicar_suavizado_frecuencia(dsa_db, sigma_freq=0):
    """
    Aplica suavizado gaussiano entre frecuencias.

    dsa_db:
        DataFrame con filas = tiempo y columnas = frecuencia.

    sigma_freq:
        Intensidad del suavizado entre columnas de frecuencia.
        - 0: sin suavizado
        - 0.5: suavizado muy leve
        - 1: suavizado moderado
        - 1.5 o 2: suavizado más fuerte
    """

    if sigma_freq == 0:
        return dsa_db.copy()

    matriz_suav = gaussian_filter1d(
        dsa_db.to_numpy(dtype=float),
        sigma=sigma_freq,
        axis=1,
        mode="nearest"
    )

    return pd.DataFrame(
        matriz_suav,
        index=dsa_db.index,
        columns=dsa_db.columns
    )


def aplicar_suavizado_temporal(dsa_db, ventana_s=0):
    """
    Aplica suavizado temporal causal.

    ventana_s:
        - 0: sin suavizado temporal
        - 5, 10, 30, 60: media móvil de los últimos N segundos.
    """

    if ventana_s == 0:
        return dsa_db.copy()

    return dsa_db.rolling(
        window=ventana_s,
        min_periods=1,
        center=False
    ).mean()


def aplicar_shift_comparacion(dsa_recon, dsa_fa, shift_s):
    """
    Aplica shift para comparar dos matrices.

    Convención:
    - shift_s > 0:
        la reconstruida se compara contra la f_a desplazada hacia delante.
    - shift_s < 0:
        la reconstruida se compara contra la f_a desplazada hacia atrás.
    - shift_s = 0:
        comparación directa.

    Devuelve dos matrices con la misma longitud.
    """

    if shift_s < 0:
        A = dsa_recon.iloc[-shift_s:].reset_index(drop=True)
        B = dsa_fa.iloc[:len(A)].reset_index(drop=True)

    elif shift_s > 0:
        A = dsa_recon.iloc[:-shift_s].reset_index(drop=True)
        B = dsa_fa.iloc[shift_s:].reset_index(drop=True)

    else:
        A = dsa_recon.reset_index(drop=True)
        B = dsa_fa.reset_index(drop=True)

    n = min(len(A), len(B))
    A = A.iloc[:n]
    B = B.iloc[:n]

    return A, B


def evaluar_dsa_reconstruida(
    nombre,
    dsa_recon_db,
    dsa_fa_db,
    mask_comun,
    sigmas_freq=(0, 0.5, 1, 1.5, 2),
    ventanas_temporales=(0, 5, 10, 30, 60),
    shifts=range(-30, 31)
):
    """
    Evalúa una DSA reconstruida frente a la DSA original f_a.

    Para cada combinación de:
    - suavizado frecuencial
    - suavizado temporal
    - shift temporal

    calcula métricas globales sobre matrices normalizadas con z-score.
    """

    resultados = []

    for sigma_freq in sigmas_freq:

        # 1) Suavizado entre frecuencias
        dsa_freq = aplicar_suavizado_frecuencia(
            dsa_recon_db,
            sigma_freq=sigma_freq
        )

        for ventana in ventanas_temporales:

            # 2) Suavizado temporal
            dsa_temp = aplicar_suavizado_temporal(
                dsa_freq,
                ventana_s=ventana
            )

            # 3) Aplicar máscara común al final
            dsa_temp_mask = dsa_temp.copy()
            dsa_temp_mask.loc[mask_comun.values, :] = np.nan

            dsa_fa_mask = dsa_fa_db.copy()
            dsa_fa_mask.loc[mask_comun.values, :] = np.nan

            for shift in shifts:

                # 4) Aplicar shift para comparar
                A, B = aplicar_shift_comparacion(
                    dsa_recon=dsa_temp_mask,
                    dsa_fa=dsa_fa_mask,
                    shift_s=shift
                )

                # 5) Asegurar mismas columnas y longitud
                A, B = fun_dsa.preparar_matrices_para_comparacion(A, B)

                try:
                    # 6) Normalizar para comparar patrones
                    A_z = fun_dsa.zscore_global(A)
                    B_z = fun_dsa.zscore_global(B)

                    # 7) Métricas
                    met = fun_dsa.comparar_dsa_global(A_z, B_z)

                    resultados.append({
                        "metodo": nombre,
                        "sigma_freq": sigma_freq,
                        "suavizado_temporal_s": ventana,
                        "shift_s": shift,
                        "n_valores": met["n_valores_comparados"],
                        "Pearson": met["Pearson"],
                        "Spearman": met["Spearman"],
                        "MAE": met["MAE"],
                        "RMSE": met["RMSE"],
                        "bias": met["bias_B_menos_A"]
                    })

                except ValueError:
                    continue

    return pd.DataFrame(resultados)

# OG

#### Archivo f_a

In [3]:
ruta_base_advanced = "../data/data_bis_advanced"

archivos_fa = fau.localizar_archivos(ruta_base_advanced, "DH*", "*.f_a")
ruta_fa = archivos_fa[0]
print("Archivo seleccionado:", ruta_fa)

tiempo_fa_unilat, dsa_unilat = fau.cargar_fa_directo(ruta_fa, escalar_db=True)

print("Dimensiones de la matriz:", dsa_unilat.shape)
print("Frecuencias:", dsa_unilat.columns.min(), "a", dsa_unilat.columns.max(), "Hz")

Se han encontrado 2 archivos *.f_a
Archivo seleccionado: ../data/data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.f_a
Dimensiones de la matriz: (2119, 60)
Frecuencias: 0.5 a 30.0 Hz


#### Archivo spa

In [4]:
ruta_spa_unilat = "../data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"


df_spa_raw = fau.procesar_spa(ruta_spa_unilat)
df_spa_unilat = fun_dsa_u.limpiar_spa_unilateral(df_spa_raw)

## Unión de los 2

In [5]:
df_merge_hor = fun_dsa.alinear_spa_con_tiempo(tiempo_fa_unilat, df_spa_unilat)
sef_hor = df_merge_hor["SEF08"]
mf_hor = df_merge_hor["MEDFRQ08"]

dsa_plot_hor, mask_total_hor = fun_dsa.preparar_dsa_con_mask(tiempo_fa_unilat, dsa_unilat, df_merge_hor)

# las matrices que vienen de la f_a suelen mostrar valores entre el 49 y 94
matriz_hor, vmin_hor, vmax_hor, norm_hor, cmap_hor = fun_dsa.preparar_escala_color_dsa(dsa_plot_hor, vmin=49, vmax=94, gamma=1)

In [12]:
cols_freq_fa = [c for c in dsa_plot_hor.columns]


print(np.nanmin(dsa_plot_hor[cols_freq_fa].values))
print(np.nanmax(dsa_plot_hor[cols_freq_fa].values))

print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 2))
print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 99.5))

55.61
103.02
70.0
96.87


# Procesado

In [7]:
archivo_r2a = r"../data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.r2a"

### Reconstrucción desde EEG crudo

In [16]:
df_eeg = fun_dsa_u.leer_r2a(
    archivo_r2a,
    fs=128,
    escala_uv=0.0511
)

# se pierde un segundo por el uso de ventanas de 2 segundos con etiqueta temporal en el centro
""" 
La función no calcula una DSA para cada muestra aislada ni para cada segundo independiente. 
Calcula una DSA por ventanas completas de 2 segundos (usando 256 muestras).
La ventana avanza cada segundo (cada 128 muestras).

Resultado: Cada fila de la DSA final no sale de un único segundo, sino de una ventana de 2 segundos.


Por eso, con ventanas de 2 segundos y paso de 1 segundo, si tienes un registro de N segundos, se obtienen aproximadamente:
N - 1 filas espectrales. 
No se obtienen N filas, porque no se puede calcular una ventana completa de 2 segundos centrada 
en todos los segundos extremos sin salirse del registro.
"""

df_dsa_canal1, frecuencias_c1 = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_1_uV",
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

df_dsa_canal2, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_2_uV",
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq = [c for c in df_dsa_canal1.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media = df_dsa_canal1.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media = (
    df_dsa_canal1[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal2[cols_freq].to_numpy(dtype=float)
) / 2

# array de los nombres de las columnas pasados a float
frecuencias_float = np.array([float(c) for c in cols_freq])

# calcular sef y mef solo para la dsa reconstruida del eeg 
# (basado en la potencia en uV)
# 1. Calculamos SEF y MEF (que tendrán 2119 o 2118 filas dependiendo de la ventana)
sef_eeg, mef_eeg = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=pot_media,
    frecuencias=frecuencias_float,
    percentil_sef=0.95,
    percentil_mef=0.50
)

# 2. Averiguamos la longitud objetivo (las filas del DataFrame de tu archivo .spa o .f_a)
# Suponiendo que tienes cargado el dF original del spa/f_a en una variable (ej: df_spa)
longitud_objetivo = len(df_spa_unilat) 
longitud_actual = len(sef_eeg)
filas_faltantes = longitud_objetivo - longitud_actual

# 3. Rellenamos dinámicamente con NaNs
if filas_faltantes > 0:
    # Creamos un array con tantos NaNs como falten (normalmente 1)
    relleno_nan = np.full(filas_faltantes, np.nan)
    
    # Los añadimos al principio
    sef_pad = np.r_[relleno_nan, sef_eeg]
    mef_pad = np.r_[relleno_nan, mef_eeg]
else:
    # Si la ventana es de 1s, filas_faltantes es 0, se queda igual
    sef_pad = sef_eeg
    mef_pad = mef_eeg

# 4. Creamos las Series finales para plotear
sef_eeg_plot = pd.Series(sef_pad, name="SEF08")
mef_eeg_plot = pd.Series(mef_pad, name="MEDFRQ08")

# --- CONVERSIÓN A DECIBELIOS (Usando la ref documentada del BIS) ---
ref_potencia = 0.0001
df_dsa_media[cols_freq] = 10 * np.log10(
    (pot_media + 1e-12) / (ref_potencia**2)
)

In [17]:
print(np.nanmin(df_dsa_media[cols_freq].values))
print(np.nanmax(df_dsa_media[cols_freq].values))

print(np.nanpercentile(df_dsa_media[cols_freq].values, 2))
print(np.nanpercentile(df_dsa_media[cols_freq].values, 99.5))

42.06825408227754
129.82281516998063
72.35147912223097
114.70939350044868


In [18]:
cols_freq_fa = [c for c in dsa_plot_hor.columns]


print(np.nanmin(dsa_plot_hor[cols_freq_fa].values))
print(np.nanmax(dsa_plot_hor[cols_freq_fa].values))

print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 2))
print(np.nanpercentile(dsa_plot_hor[cols_freq_fa].values, 99.5))

55.61
103.02
70.0
96.87


### Adaptación temporal, máscara y plot de DSA EEG

In [19]:
df_spa_unilat, hora_inicio = fun_dsa.obtener_hora_inicio_desde_spa(
    df_spa_unilat
)

tiempo_eeg, dsa_eeg = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_media,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

df_merge_plot = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg,
    df_spa=df_spa_unilat
)

df_merge_plot["SEF08"] = sef_eeg_plot.values
df_merge_plot["MEDFRQ08"] = mef_eeg_plot.values

_, mask_total = fun_dsa.preparar_dsa_con_mask(
    tiempo=tiempo_eeg,
    dsa=dsa_eeg,
    df_merge=df_merge_plot,
    umbral_sqi=15,
    umbral_ceros=0.9
)

mask_comun = mask_total.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color


In [20]:
# DSA original f_a con máscara común

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_unilat.copy()
dsa_fa_plot.loc[mask_comun.values, :] = np.nan



# DSA reconstruida directa con máscara común

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_directa_plot = dsa_eeg.copy()
dsa_eeg_directa_plot.loc[mask_comun.values, :] = np.nan

In [21]:
pot_c1 = df_dsa_canal1[cols_freq].to_numpy(dtype=float)
pot_c2 = df_dsa_canal2[cols_freq].to_numpy(dtype=float)

opciones_canales = {
    "solo_canal_1": pot_c1,
    "solo_canal_2": pot_c2,
    "media_c1_c2": (pot_c1 + pot_c2) / 2,
    "maximo_c1_c2": np.maximum(pot_c1, pot_c2),
    "ponderada_70_30": 0.7 * pot_c1 + 0.3 * pot_c2,
    "ponderada_30_70": 0.3 * pot_c1 + 0.7 * pot_c2,
}

In [22]:
def construir_dsa_db_desde_potencia(
    pot_matrix,
    df_template,
    cols_freq,
    frecuencias,
    hora_inicio,
    ref_uv_rms=0.0001,
    insertar_fila_inicial_nan=True
):
    """
    Convierte una matriz de potencia/densidad lineal a DSA en dB
    y la adapta al eje temporal absoluto.

    pot_matrix:
        matriz tiempo x frecuencia en escala lineal.

    df_template:
        DataFrame con columna tiempo_s y columnas de frecuencia,
        usado como plantilla de estructura.

    cols_freq:
        columnas de frecuencia.

    frecuencias:
        array de frecuencias.

    hora_inicio:
        hora inicial absoluta del registro.

    ref_uv_rms:
        referencia usada para la conversión a dB.
    """

    df_tmp = df_template.copy()

    df_tmp[cols_freq] = 10 * np.log10(
        (pot_matrix + 1e-12) / (ref_uv_rms ** 2)
    )

    tiempo_tmp, dsa_tmp = fun_dsa.adaptar_dsa_reconstruida_para_plot(
        df_dsa=df_tmp,
        frecuencias=frecuencias,
        hora_inicio=hora_inicio,
        insertar_fila_inicial_nan=insertar_fila_inicial_nan
    )

    return tiempo_tmp, dsa_tmp

In [ ]:
todos_resultados_canales = []

dsa_canales_dict = {}

for nombre_opcion, pot_opcion in opciones_canales.items():

    print(f"Procesando: {nombre_opcion}")

    tiempo_tmp, dsa_tmp = construir_dsa_db_desde_potencia(
        pot_matrix=pot_opcion,
        df_template=df_dsa_canal1,
        cols_freq=cols_freq,
        frecuencias=frecuencias_c1,
        hora_inicio=hora_inicio,
        ref_uv_rms=0.0001,
        insertar_fila_inicial_nan=False
    )

    dsa_canales_dict[nombre_opcion] = dsa_tmp

    df_res = evaluar_dsa_reconstruida(
        nombre=nombre_opcion,
        dsa_recon_db=dsa_tmp,
        dsa_fa_db=dsa_unilat,
        mask_comun=mask_comun,
        sigmas_freq=[0, 0.5, 1, 1.5, 2],
        ventanas_temporales=[0, 5, 10, 30, 60],
        shifts=range(-30, 31)
    )

    todos_resultados_canales.append(df_res)

resultados_canales = pd.concat(
    todos_resultados_canales,
    ignore_index=True
)

resultados_canales.sort_values("Pearson", ascending=False).head(20)

Procesando: solo_canal_1


In [ ]:
mejores_por_metodo = (
    resultados_canales
    .sort_values("Pearson", ascending=False)
    .groupby("metodo", as_index=False)
    .first()
    .sort_values("Pearson", ascending=False)
)

mejores_por_metodo

In [ ]:
mejor_global = resultados_canales.sort_values("Pearson", ascending=False).iloc[0]

mejor_global

In [ ]:
mejor_metodo = mejor_global["metodo"]
mejor_sigma_freq = mejor_global["sigma_freq"]
mejor_suavizado_temporal = int(mejor_global["suavizado_temporal_s"])
mejor_shift = int(mejor_global["shift_s"])

print("Mejor método:", mejor_metodo)
print("Sigma frecuencia:", mejor_sigma_freq)
print("Suavizado temporal:", mejor_suavizado_temporal)
print("Shift:", mejor_shift)
print("Pearson:", mejor_global["Pearson"])
print("Spearman:", mejor_global["Spearman"])
print("MAE:", mejor_global["MAE"])
print("RMSE:", mejor_global["RMSE"])

In [ ]:
# Recuperar la DSA base correspondiente al mejor método
dsa_mejor_base = dsa_canales_dict[mejor_metodo].copy()

# Suavizado en frecuencia
dsa_mejor_freq = aplicar_suavizado_frecuencia(
    dsa_mejor_base,
    sigma_freq=mejor_sigma_freq
)

# Suavizado temporal
dsa_mejor_temp = aplicar_suavizado_temporal(
    dsa_mejor_freq,
    ventana_s=mejor_suavizado_temporal
)

# Mantener duración completa y aplicar shift con NaN
dsa_mejor_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_mejor_temp.index,
    columns=dsa_mejor_temp.columns
)

if mejor_shift > 0:
    dsa_mejor_shift_full.iloc[mejor_shift:, :] = (
        dsa_mejor_temp.iloc[:-mejor_shift, :].to_numpy()
    )

elif mejor_shift < 0:
    dsa_mejor_shift_full.iloc[:mejor_shift, :] = (
        dsa_mejor_temp.iloc[-mejor_shift:, :].to_numpy()
    )

else:
    dsa_mejor_shift_full.iloc[:, :] = dsa_mejor_temp.to_numpy()

# Aplicar máscara común al final
dsa_mejor_shift_full.loc[mask_comun.values, :] = np.nan

# Preparar escala de color
matriz_mejor, vmin_mejor, vmax_mejor, norm_mejor, cmap_mejor = fun_dsa.preparar_escala_color_dsa(
    dsa_mejor_shift_full,
    gamma=0.25
)

In [ ]:
fig_mejor, ax_mejor, ax_band_mejor, cax_mejor = fun_plot.plot_dsa_con_sef_mef(
    tiempo=tiempo_eeg,
    frecuencias=frecuencias_c1,
    matriz=matriz_mejor,
    norm=norm_mejor,
    cmap=cmap_mejor,
    df_merge=None,
    titulo=f"Mejor DSA reconstruida: {mejor_metodo}, sigma={mejor_sigma_freq}, suav={mejor_suavizado_temporal}s, shift={mejor_shift}s",
    etiqueta_colorbar="Intensidad espectral (dB)",
    mask_total=mask_comun,
    mostrar_sef=False,
    mostrar_mef=False
)

### Comprobar que ambas matrices están alineadas

In [ ]:
print("DSA f_a:", dsa_fa_plot.shape)
print("DSA EEG:", dsa_eeg_directa_plot.shape)

print("¿Tiempos iguales?")
print((tiempo_fa_unilat.reset_index(drop=True) == tiempo_eeg.reset_index(drop=True)).all())

print("Rango DSA EEG reconstruida:")
print(np.nanmin(dsa_eeg_directa_plot.values), np.nanmax(dsa_eeg_directa_plot.values))

print("Rango DSA f_a:")
print(np.nanmin(dsa_fa_plot.values), np.nanmax(dsa_fa_plot.values))

## Preparar escalas de color de f_a y EEG directa

In [ ]:
matriz_fa, vmin_fa, vmax_fa, norm_fa, cmap_fa = fun_dsa.preparar_escala_color_dsa(
    dsa_fa_plot,
    vmin=49,
    vmax=94,
    gamma=1
)

matriz_eeg, vmin_eeg, vmax_eeg, norm_eeg, cmap_eeg = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_directa_plot,
    gamma=0.4
)

### Comparación base

In [ ]:
dsa_eeg_comparacion, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot,
    dsa_fa_plot
)

dsa_eeg_z = fun_dsa.zscore_global(dsa_eeg_comparacion)
dsa_fa_z = fun_dsa.zscore_global(dsa_fa_comparacion)

metricas_base = fun_dsa.comparar_dsa_global(
    dsa_eeg_z,
    dsa_fa_z
)

metricas_base

### Correlación por frecuencia

In [ ]:
df_corr_freq = fun_dsa.correlacion_por_frecuencia(
    dsa_eeg_z,
    dsa_fa_z
)

"""plt.figure(figsize=(12, 4))

plt.plot(
    df_corr_freq["frecuencia_Hz"],
    df_corr_freq["correlacion"],
    marker="o"
)

for f in [4, 8, 13]:
    plt.axvline(f, color="gray", linestyle="--", linewidth=1, alpha=0.6)

plt.axhline(0, color="gray", linestyle="--", linewidth=1)

plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Correlación")
plt.title("Correlación por frecuencia entre DSA EEG y DSA f_a")
plt.tight_layout()
plt.show()"""

### Suavizado temporal

Suavizado temporal de la DSA reconstruida desde EEG

La DSA reconstruida desde el EEG crudo viene de un cálculo directo y produce una DSA con muchos cambios rápidos. 

La DSA del f_a del BIS se ve más suave y no cambia tan bruscamente de segundo a segundo. Sugiere que el BIS podría aplicar algún tipo de promedio temporal interno antes de guardar el f_a.

### Suavizado en frecuencia

In [ ]:
sigmas_freq = [0, 0.5, 1, 1.5, 2]
ventanas_temporales = [1, 5, 10, 30, 60]
shifts = range(-30, 31)

resultados_freq = evaluar_dsa_reconstruida(
    nombre="EEG media canales + suavizado frecuencia",
    dsa_recon_db=dsa_eeg,
    dsa_fa_db=dsa_unilat,
    mask_comun=mask_comun,
    sigmas_freq=sigmas_freq,
    ventanas_temporales=ventanas_temporales,
    shifts=shifts
)

resultados_freq.sort_values("Pearson", ascending=False).head(15)

In [ ]:
resultados_freq.sort_values("Spearman", ascending=False).head(15)

### Suavizado + shift

In [ ]:
ventanas_sp_smooth = [5, 10, 30, 60]

df_suav_shift = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comparacion,
    dsa_fa_comparacion,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=range(-60, 61)
)

df_suav_shift.sort_values("Pearson", ascending=False).head(7)

In [ ]:
df_suav_shift.sort_values("Spearman", ascending=False).head(7)

### Tabla resumen final

La variante que mejor aproximó la DSA reconstruida desde EEG crudo a la DSA del archivo `f_a` fue la media de potencias de los dos canales, calculada a partir de densidad espectral de potencia y convertida posteriormente a dB. 

Tras aplicar un suavizado temporal causal de 30 s y un desplazamiento temporal de 10 u 11 s, se obtuvo una correlación de Pearson de aproximadamente 0.706 y una correlación de Spearman de aproximadamente 0.704. 

Esto sugiere que el archivo `f_a` incorpora información espectral de ambos canales, así como procesamiento temporal suavizado y un posible retardo asociado al cálculo interno del monitor BIS.

In [ ]:
mejor_pearson = df_suav_shift.sort_values("Pearson", ascending=False).iloc[0]
mejor_spearman = df_suav_shift.sort_values("Spearman", ascending=False).iloc[0]

df_resumen_final = pd.DataFrame([
    {
        "criterio": "Mejor Pearson",
        "suavizado_s": mejor_pearson["suavizado_s"],
        "shift_s": mejor_pearson["shift_s"],
        "Pearson": mejor_pearson["Pearson"],
        "Spearman": mejor_pearson["Spearman"],
        "MAE": mejor_pearson["MAE"],
        "RMSE": mejor_pearson["RMSE"],
    },
    {
        "criterio": "Mejor Spearman",
        "suavizado_s": mejor_spearman["suavizado_s"],
        "shift_s": mejor_spearman["shift_s"],
        "Pearson": mejor_spearman["Pearson"],
        "Spearman": mejor_spearman["Spearman"],
        "MAE": mejor_spearman["MAE"],
        "RMSE": mejor_spearman["RMSE"],
    }
])

df_resumen_final

## Suavizado limpio

In [ ]:
suavizado_final = 30

dsa_eeg_suav = dsa_eeg.copy().rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

In [ ]:

# máscara común al final
dsa_eeg_suav_plot = dsa_eeg_suav.copy()
dsa_eeg_suav_plot.loc[mask_comun.values, :] = np.nan

matriz_opt, vmin_opt, vmax_opt, norm_opt, cmap_opt = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_plot,
    gamma=0.25
)

## Comprobación de bandas blancas

In [ ]:
mask_blanca_directa = dsa_eeg_directa_plot.isna().all(axis=1)
mask_blanca_suav = dsa_eeg_suav_plot.isna().all(axis=1)

print("bandas blancas directa vs suavizada")
print((mask_blanca_directa == mask_blanca_suav).all())

print("Diferencias:")
print((mask_blanca_directa != mask_blanca_suav).sum())

## Al haber recortado 10 segundos


In [ ]:
shift_final = 10

# matriz vacía del mismo tamaño que la DSA suavizada original
dsa_eeg_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_suav.index,
    columns=dsa_eeg_suav.columns
)

# Desplazar la DSA suavizada hacia delante:
# las filas originales 0:-shift pasan a ocupar las filas shift:
dsa_eeg_suav_shift_full.iloc[shift_final:, :] = dsa_eeg_suav.iloc[:-shift_final, :].to_numpy()

# Aplicar la máscara común sobre la línea temporal original completa
dsa_eeg_suav_shift_full.loc[mask_comun.values, :] = np.nan

# Preparar escala de color
matriz_opt_desfase_full, vmin_opt_desfase_full, vmax_opt_desfase_full, norm_opt_desfase_full, cmap_opt_desfase_full = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_suav_shift_full,
    gamma=0.25
)

print("Tiempo:", len(tiempo_fa_unilat))
print("f_a:", matriz_hor.shape)
print("shift full:", matriz_opt_desfase_full.shape)

## Comprobación forma DSA

In [ ]:
print("DSA f_a:", dsa_fa_plot.shape)
print("DSA EEG:", dsa_eeg_directa_plot.shape)
print("DSA suavizada:", dsa_eeg_suav_plot.shape)
print("DSA shift:", dsa_eeg_suav_shift_full.shape)

print("¿Tiempos iguales?")
print((tiempo_fa_unilat.reset_index(drop=True) == tiempo_eeg.reset_index(drop=True)).all())


print("Rango DSA EEG reconstruida:")
print(np.nanmin(dsa_eeg_directa_plot.values), np.nanmax(dsa_eeg_directa_plot.values))

print("Rango DSA f_a:")
print(np.nanmin(dsa_fa_plot.values), np.nanmax(dsa_fa_plot.values))

# COMPARACIÓN FINAL

In [ ]:
paneles = [
    
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_eeg,
        "norm": norm_eeg,
        "cmap": cmap_eeg,
        "df_merge": df_merge_plot,
        "titulo": "DSA reconstruida desde EEG crudo",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt,
        "norm": norm_opt,
        "cmap": cmap_opt,
        "df_merge": df_merge_plot,
        "titulo": "DSA reconstruida suavizada",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_fa_unilat,
        "frecuencias": dsa_fa_plot.columns.astype(float),
        "matriz": matriz_fa,
        "norm": norm_fa,
        "cmap": cmap_fa,
        "df_merge": df_merge_hor,
        "titulo": "DSA original desde archivo f_a",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt_desfase_full,
        "norm": norm_opt_desfase_full,
        "cmap": cmap_opt_desfase_full,
        "df_merge": df_merge_plot,
        "titulo": "DSA suavizada + shift exploratorio",
        "etiqueta_colorbar": "Intensidad espectral (dB)",
        "mask_total": mask_comun
    }
]

fig_grid, axes_grid = fun_dsa.plot_cuadricula_4_dsa(
    paneles,
    titulo_general="Comparación visual de las cuatro matrices DSA"
)

# Cálculo de SEF / MEF propios

In [ ]:
# ============================================================
# SEF / MEF propios para cada representación
# ============================================================

suavizado_final=30

# 1. SEF/MEF de referencia del BIS, extraídos del .spa
sef_fa_spa = df_merge_hor["SEF08"].copy()
mef_fa_spa = df_merge_hor["MEDFRQ08"].copy()


# 2. SEF/MEF de la DSA reconstruida directa desde EEG
frecuencias_float = np.array([float(c) for c in cols_freq])

sef_eeg, mef_eeg = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=pot_media,
    frecuencias=frecuencias_float,
    percentil_sef=0.95,
    percentil_mef=0.50
)

sef_eeg_plot = pd.Series(np.r_[sef_eeg], name="SEF08")
mef_eeg_plot = pd.Series(np.r_[mef_eeg], name="MEDFRQ08")

df_merge_eeg = df_merge_plot.copy()
df_merge_eeg["SEF08"] = sef_eeg_plot.values
df_merge_eeg["MEDFRQ08"] = mef_eeg_plot.values


# 3. SEF/MEF de la DSA reconstruida suavizada
df_pot_media = pd.DataFrame(pot_media, columns=cols_freq)

df_pot_media_suav = df_pot_media.rolling(
    window=suavizado_final,
    min_periods=1,
    center=False
).mean()

sef_eeg_suav, mef_eeg_suav = fun_dsa.calcular_sef_mef_desde_potencia(
    potencia=df_pot_media_suav.to_numpy(dtype=float),
    frecuencias=frecuencias_float,
    percentil_sef=0.95,
    percentil_mef=0.50
)

sef_eeg_suav_plot = pd.Series(np.r_[sef_eeg_suav], name="SEF08")
mef_eeg_suav_plot = pd.Series(np.r_[mef_eeg_suav], name="MEDFRQ08")

df_merge_suav = df_merge_plot.copy()
df_merge_suav["SEF08"] = sef_eeg_suav_plot.values
df_merge_suav["MEDFRQ08"] = mef_eeg_suav_plot.values


# 4. SEF/MEF de la DSA suavizada + shift, manteniendo duración original
df_merge_suav_shift_full = df_merge_suav.copy()

df_merge_suav_shift_full["SEF08"] = np.nan
df_merge_suav_shift_full["MEDFRQ08"] = np.nan

df_merge_suav_shift_full.loc[shift_final:, "SEF08"] = sef_eeg_suav_plot.iloc[:-shift_final].to_numpy()
df_merge_suav_shift_full.loc[shift_final:, "MEDFRQ08"] = mef_eeg_suav_plot.iloc[:-shift_final].to_numpy()

In [ ]:
paneles = [
    
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_eeg,
        "norm": norm_eeg,
        "cmap": cmap_eeg,
        "df_merge": df_merge_eeg,
        "titulo": "DSA reconstruida desde EEG crudo",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt,
        "norm": norm_opt,
        "cmap": cmap_opt,
        "df_merge": df_merge_suav,
        "titulo": "DSA reconstruida suavizada",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_fa_unilat,
        "frecuencias": dsa_fa_plot.columns.astype(float),
        "matriz": matriz_fa,
        "norm": norm_fa,
        "cmap": cmap_fa,
        "df_merge": df_merge_hor,
        "titulo": "DSA original desde archivo f_a",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    },
    {
        "tiempo": tiempo_eeg,
        "frecuencias": frecuencias_c1,
        "matriz": matriz_opt_desfase_full,
        "norm": norm_opt_desfase_full,
        "cmap": cmap_opt_desfase_full,
        "df_merge": df_merge_suav_shift_full,
        "titulo": "DSA suavizada + shift temporal",
        "mask_total": mask_comun,
        "mostrar_sef": True,
        "mostrar_mef": True
    }
]


fig_grid, axes_grid = fun_dsa.plot_cuadricula_4_dsa(
    paneles,
    titulo_general="Comparación visual de las cuatro matrices DSA"
)